<a href="https://colab.research.google.com/github/hasnainaliasghar/ai-journey/blob/main/08%20Pytorch/RNN/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [115]:
import pandas as pd
from google.colab import drive

In [116]:


import gspread
from google.colab import auth
from google.auth import default
# 1. Authenticate user
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Open the Google Sheet by name or URL
# You can use the document name 'Dataset' or its full URL
spreadsheet = gc.open('Dataset')

# 3. Select the worksheet tab (e.g., '100_Unique_QA_Dataset')
worksheet = spreadsheet.worksheet('100_Unique_QA_Dataset')

# 4. Convert records to a DataFrame
data = worksheet.get_all_records()
df = pd.DataFrame(data)

# Display the first few rows
df.tail()

,question,answer
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango
89,Which country is known for the Eiffel Tower?,France
90,Who is Hasnain?,Person
91,What is yogurt?,BacterailFood


In [117]:
#tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace("?", "")
  text = text.replace("'","")
  return text.split()

tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [118]:
#vocab
vocab = {'<UNK>': 0}

In [119]:
print(df["question"][0])
print(df["answer"][0])

What is the capital of France?
Paris


In [120]:
def build_vocab(row):

  tokenize_question = tokenize(str(row["question"]))
  tokenize_answer = tokenize(str(row["answer"]))
  merged_tokens = tokenize_question + tokenize_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
87,None
88,None
89,None
90,None


In [121]:
len(vocab)

327

In [122]:
#convert words to numeric indeces
def text_to_indices(text, vocab):
  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [123]:
text_to_indices("Who is Hasnain", vocab)

[10, 2, 324]

In [124]:
import torch
from torch.utils.data import Dataset, DataLoader

In [125]:
class QADataset(Dataset):

  def __init__ (self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    numerical_question = text_to_indices(str(self.df.iloc[idx]['question']), self.vocab)
    numerical_answer = text_to_indices(str(self.df.iloc[idx]['answer']), self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [126]:
dataset = QADataset(df, vocab)

In [127]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [128]:
import torch.nn as nn

In [129]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50,64, batch_first=True)
    self.fc = nn.Linear(64,vocab_size)

  def forward(self,question):
    embedded_question = self.embedding(question)
    hidden, final= self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))
    return output

In [130]:
learning_rate = 0.001
epochs = 21

In [131]:
model = SimpleRNN(len(vocab))

In [132]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [133]:
for epoch in range(epochs):
  total_loss = 0

  for question, answer in dataloader:
    optimizer.zero_grad()

    output = model(question)

    # Squeeze the target 'answer' to be 1D (selecting the first token index for single-word prediction)
    loss = criterion(output, answer[:, 0])
    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch}, Loss: {total_loss: .4f}")

Epoch: 0, Loss:  540.8071
Epoch: 1, Loss:  470.0649
Epoch: 2, Loss:  389.7978
Epoch: 3, Loss:  326.6838
Epoch: 4, Loss:  273.5821
Epoch: 5, Loss:  223.7278
Epoch: 6, Loss:  178.0933
Epoch: 7, Loss:  137.8887
Epoch: 8, Loss:  105.0903
Epoch: 9, Loss:  79.1601
Epoch: 10, Loss:  60.5432
Epoch: 11, Loss:  46.8007
Epoch: 12, Loss:  36.8358
Epoch: 13, Loss:  29.8525
Epoch: 14, Loss:  24.3319
Epoch: 15, Loss:  20.2306
Epoch: 16, Loss:  17.2344
Epoch: 17, Loss:  14.6464
Epoch: 18, Loss:  12.9786
Epoch: 19, Loss:  10.8031
Epoch: 20, Loss:  9.5105


In [134]:
def predict(model, question, threshold = 0.5):
  # convert Question Numbers
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  output = model(question_tensor)

  probs = torch.nn.functional.softmax(output, dim=1)
  value, index =torch.max(probs, dim=1)

  if value < threshold:
    print("I dont Know")

  print(list(vocab.keys())[index])

In [137]:
predict(model, "What is yogurt")

bacterailfood
